# Stroke priority modeling

Binary classification for stroke calls (demand code `54`) using `p1_real_emerg_bps` as the outcome. Model selection optimizes average precision, and the decision threshold is estimated from out-of-fold predictions using Youden's J statistic.

## Environment

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from kbase.config import settings
import kbase.modeling as ml 
import kbase.eda as eda 

## AutoML time-budget comparison

Run the same out-of-fold AutoML experiment with increasing search budgets. These runs omit feature-importance calculations so that the comparison focuses on model search.

### 10-minute AutoML search

In [ ]:
ml.run_binary_automl_model(
    cohort_name = "Stroke",
    target_column = "p1_real_emerg_bps",
    demand_code = 54,
    triage_value = None,
    export_table = True,
    time_budget = 600,
    include_com = False,
    optimize_metric="ap",
    train_threshold = "oof",
    use_youden = True,
    optimize_beta = 1, 
    feature_importance = 0
)

### 15-minute AutoML search

In [ ]:
ml.run_binary_automl_model(
    cohort_name = "Stroke",
    target_column = "p1_real_emerg_bps",
    demand_code = 54,
    triage_value = None,
    export_table = True,
    time_budget = 900,
    include_com = False,
    optimize_metric="ap",
    train_threshold = "oof",
    use_youden = True,
    optimize_beta = 1, 
    feature_importance = 0
)

### 20-minute AutoML search

In [ ]:
ml.run_binary_automl_model(
    cohort_name = "Stroke",
    target_column = "p1_real_emerg_bps",
    demand_code = 54,
    triage_value = None,
    export_table = True,
    time_budget = 1200,
    include_com = False,
    optimize_metric="ap",
    train_threshold = "oof",
    use_youden = True,
    optimize_beta = 1, 
    feature_importance = 0
)

### 25-minute AutoML search

In [ ]:
ml.run_binary_automl_model(
    cohort_name = "Stroke",
    target_column = "p1_real_emerg_bps",
    demand_code = 54,
    triage_value = None,
    export_table = True,
    time_budget = 1500,
    include_com = False,
    optimize_metric="ap",
    train_threshold = "oof",
    use_youden = True,
    optimize_beta = 1, 
    feature_importance = 0
)

### 30-minute AutoML search

In [ ]:
ml.run_binary_automl_model(
    cohort_name = "Stroke",
    target_column = "p1_real_emerg_bps",
    demand_code = 54,
    triage_value = None,
    export_table = True,
    time_budget = 1800,
    include_com = False,
    optimize_metric="ap",
    train_threshold = "oof",
    use_youden = True,
    optimize_beta = 1, 
    feature_importance = 0
)

### 60-minute AutoML search

In [ ]:
ml.run_binary_automl_model(
    cohort_name = "Stroke",
    target_column = "p1_real_emerg_bps",
    demand_code = 54,
    triage_value = None,
    export_table = True,
    time_budget = 3600,
    include_com = False,
    optimize_metric="ap",
    train_threshold = "oof",
    use_youden = True,
    optimize_beta = 1, 
    feature_importance = 0
)

## Pipeline smoke test

Use this short 100-second run to validate the end-to-end pipeline, including interpretability outputs, before launching longer experiments.

In [ ]:
ml.run_binary_automl_model(
    cohort_name = "Stroke",
    target_column = "p1_real_emerg_bps",
    demand_code = 54,
    triage_value = None,
    export_table = True,
    time_budget = 100,
    include_com = False,
    optimize_metric="ap",
    train_threshold = "oof",
    use_youden = True,
    optimize_beta = 1, 
    feature_importance = 2
)

## Time-budget and iteration-limit comparison

First run a 30-minute search and record the number of FLAML iterations completed. Then replay the search with that iteration count and no time limit to compare both stopping strategies.

In [ ]:
result = ml.run_binary_automl_model(
      cohort_name="Stroke",
      target_column="p1_real_emerg_bps",
      demand_code=54,
      triage_value=None,
      export_table=True,
      time_budget=1800,
      max_iter=None,
      include_com=False,
      optimize_metric="ap",
      train_threshold="oof",
      use_youden=True,
      optimize_beta=1,
      feature_importance=0,
      plot_shap=False,
      n_interpretability_features=0,
  )

automl = result["automl"]

n_iter_done = getattr(automl, "_track_iter", None)
if n_iter_done is not None:
    n_iter_done += 1

print("Total FLAML iterations:", n_iter_done)
max_iter = n_iter_done
max_iter


In [ ]:
result_iter = ml.run_binary_automl_model(
      cohort_name="Stroke",
      target_column="p1_real_emerg_bps",
      demand_code=54,
      triage_value=None,
      export_table=True,
      time_budget=-1,
      max_iter=max_iter,
      include_com=False,
      optimize_metric="ap",
      train_threshold="oof",
      use_youden=True,
      optimize_beta=1,
      feature_importance=0,
      plot_shap=False,
      n_interpretability_features=0,
  )


result["summary"]

result_iter["summary"]

result["automl"].best_estimator
result["automl"].best_config

result_iter["automl"].best_estimator
result_iter["automl"].best_config

## Final fixed model

### Selected LightGBM configuration

Reproduce the selected stroke model with the stored LightGBM configuration, export the reduced modeling table, and generate the full feature-importance output.

In [ ]:
ml.run_binary_automl_model(
    cohort_name="Stroke",
    target_column="p1_real_emerg_bps",
    demand_code=54,
    time_budget=-1,
    fixed_config=ml.FIXED_LGBM_STROKE_HIST_P1_REAL_EMERG_BPS_20260821,
    fixed_estimator="lgbm",
    include_com=False,
    optimize_metric="ap",
    train_threshold="oof",
    use_youden=True,
    export_table=True,
    feature_importance=2
)